# Final semantic analysis (RQ2) - canonical thesis results

The semantic layer of the justification analysis: **what types of information
does each model invoke in the justification it states for its vote?**

The input is the merged DeepSeek annotation run for the active stage,
resolved by the shared `AnalysisConfig`. For the base stage that is the frozen
run: 2,292 justifications, 3 models x 191 games x 4 runs, annotated by DeepSeek V4
Pro against the frozen 8-category schema. Nothing in this notebook re-annotates
or edits that file.

### What this analysis does and does not establish

It is **descriptive and associational throughout**. It measures the semantic
content of *stated* justifications. It does **not** establish internal model
reasoning, causal effects of a reasoning type, faithfulness of the explanation
to the computation that produced the vote, or the logical soundness of the
evidence cited.

### Design rules, applied everywhere

| rule | why |
|---|---|
| the primary unit is the **justification**, not the sentence | models were prompted for 3-5 sentences; repeating a category is a length artefact, not a stronger signal |
| each run is computed independently, then mean +/- SD across the three stochastic runs | a run is a realisation, not a sample of justifications |
| **stochastic and greedy are never pooled** | greedy is one deterministic run; its SD is undefined, not zero |
| every bootstrap resamples **games**, carrying all runs of a game together | the three stochastic runs of a game are repeated realisations of one transcript, not independent games |
| model comparisons reuse the **same resampled game ids** across models | pairs the comparison, cancelling game-level variation |
| 10,000 replicates, seed `20260826`, 95% percentile intervals, **no p-values** | intervals are descriptive uncertainty, not a decision rule |

All computation lives in `src/justification_analysis/semantic/`; this notebook
orchestrates and inspects it.

## 1. Setup and paths

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while current.name != repo_name:
        if current.parent == current:
            raise FileNotFoundError(f"repo root {repo_name!r} not found")
        current = current.parent
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.semantic import semantic_final as sem
from src.justification_analysis.semantic import semantic_figures as figs
from src.justification_analysis.pipeline import config as pipeline_config

# --- the one thing to change for a fine-tuned rerun ---------------------
CONFIG = pipeline_config.default_config(stage="base", repo_root=REPO_ROOT)

FINAL_TABLES = sem.final_tables_dir(CONFIG)
FINAL_FIGURES = sem.final_figures_dir(CONFIG)
FINAL_TABLES.mkdir(parents=True, exist_ok=True)
FINAL_FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)

print("stage  :", CONFIG.stage)
print("input  :", sem.annotations_path(CONFIG).relative_to(REPO_ROOT))
print("tables :", FINAL_TABLES.relative_to(REPO_ROOT))
print("figures:", FINAL_FIGURES.relative_to(REPO_ROOT))
print()
print("categories :", ", ".join(sem.CATEGORY_ORDER), f"(+ {sem.OTHER_CATEGORY})")
print("models     :", ", ".join(sem.MODEL_ORDER))
print(f"bootstrap  : {sem.BOOTSTRAP_REPLICATES:,} replicates, seed {sem.BOOTSTRAP_SEED}")

stage  : base
input  : results\justification_annotation\full_frozen\annotations.jsonl
tables : analysis\cross_model\base\voting\prompt_v4\justification_analysis\semantic_annotation\thesis_tables\final_semantic
figures: analysis\cross_model\base\voting\prompt_v4\justification_analysis\semantic_annotation\figures\final_semantic

categories : Mechanical, Testimony, SocialJudgment, Behavioral, ClaimComparison, Payoff, Uncertainty (+ Other)
models     : Gemma 4 2B, Gemma 4 4B, Gemma 4 31B
bootstrap  : 10,000 replicates, seed 20260826


## 2. Load annotations

`load_annotations` returns three tidy frames plus a repair log. The sentence
text is taken from the annotation file, except where it disagrees with the text
that was actually sent to the annotator - see section 3.

In [2]:
data = sem.load_annotations(config=CONFIG)

for name, frame in data.items():
    print(f"{name:16s} {frame.shape}")

data["justifications"].head(3)

justifications   (2292, 22)
sentences        (8044, 5)
labels           (11526, 6)
repairs          (1, 7)


,justification_id,model_key,model,run_label,decoding_group,game_id,is_correct,vote,n_sentences,n_sentences_metadata,n_labels,n_validation_flags,has_Mechanical,has_Testimony,has_SocialJudgment,has_Behavioral,has_ClaimComparison,has_Payoff,has_Uncertainty,has_Other,n_distinct_categories,has_any_substantive
0,2B__Ego4D__0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae...,2B,Gemma 4 2B,run_1,Stochastic,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,False,Hailey,3,3,4,0,False,False,False,True,False,True,True,False,3,True
1,2B__Ego4D__0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae...,2B,Gemma 4 2B,run_1,Stochastic,Ego4D / 0a6ef9dc-a2dc-452b-a907-d6fa2ed4cae0 /...,False,No Werewolf,3,3,4,0,False,False,False,False,False,True,True,False,2,True
2,2B__Ego4D__0c2659db-7bd4-4b37-9b08-4e247befe38...,2B,Gemma 4 2B,run_1,Stochastic,Ego4D / 0c2659db-7bd4-4b37-9b08-4e247befe382 /...,False,Elliot,2,2,5,0,True,True,False,False,False,True,False,False,3,True


## 3. Integrity / cleaning checks

Every count is recomputed from the file. Nothing here trusts a previously
reported number: expectations are the frozen invariants, and a disagreement
shows up as `FAIL`.

In [3]:
integrity = sem.integrity_summary(data, REPO_ROOT)
display(integrity)

assert not (integrity["status"] == "FAIL").any(), "an integrity check failed"
print("no integrity check failed")

,check,observed,expected,status
0,justifications,2292,2292,OK
1,unique justification ids,2292,2292,OK
2,duplicate justification ids,0,0,OK
3,games,191,191,OK
4,models,3,3,OK
5,runs,4,4,OK
6,justifications per model,[764],[764],OK
7,justifications per model x run,[191],[191],OK
8,model x game x run fully crossed,2292,2292,OK
9,sentences,8044,8044,OK


no integrity check failed


### 3.1 The repaired sentence

The annotator was asked to echo each sentence back verbatim. Exactly one
sentence in 8,044 came back altered, and it is repaired here from the input
shard, which is the authority for what was sent.

In [4]:
repairs = data["repairs"]
print(f"{len(repairs)} sentence(s) repaired\n")
for row in repairs.itertuples():
    print("justification:", row.justification_id)
    print("sentence     :", row.sentence_id)
    print("as annotated :", repr(row.annotated_text))
    print("as sent      :", repr(row.source_text))
    print("labels on this sentence:", row.n_labels_on_sentence)
    print("evidence spans verbatim before/after repair:",
          row.spans_verbatim_before, "/", row.spans_verbatim_after)

1 sentence(s) repaired

justification: 4B__Youtube__ONE#NIGHT#ULTIMATE#WEREWOLF#28##February#17th#2018__Game1__run_2
sentence     : 2
as annotated : '"I want to win\' (130).'
as sent      : "I want to win' (130)."
labels on this sentence: 0
evidence spans verbatim before/after repair: 0 / 0


### 3.2 Evidence spans

51 of 11,526 assignments carry an `evidence_span` that is not a literal
substring of its sentence - the annotator paraphrased or stitched fragments.
**The category labels stay in**: the failure is in the span, not the
classification. No analysis in this notebook is span-level, so nothing is
excluded; the count is recorded so the limitation is on the record.

In [5]:
labels = data["labels"]
non_verbatim = labels.loc[~labels["span_is_verbatim"]]

print(f"non-verbatim spans: {len(non_verbatim)} / {len(labels)} "
      f"({100 * len(non_verbatim) / len(labels):.2f}%)")
print(f"affected justifications: {non_verbatim['justification_id'].nunique()}")
print()
print("by category:")
display(
    non_verbatim["category"].value_counts().rename("n_non_verbatim").to_frame()
    .join(labels["category"].value_counts().rename("n_total"))
    .assign(pct=lambda f: 100 * f["n_non_verbatim"] / f["n_total"])
    .round(2)
)

non-verbatim spans: 51 / 11526 (0.44%)
affected justifications: 46

by category:


,n_non_verbatim,n_total,pct
category,,,
Testimony,26,3335,0.78
SocialJudgment,10,1951,0.51
Behavioral,9,1690,0.53
ClaimComparison,4,955,0.42
Uncertainty,1,1328,0.08
Payoff,1,1650,0.06


### 3.3 Multi-label distribution

An earlier quick summary quoted 45% / 34% / 8%, which does not sum to 100
because it mixed two denominators. They answer different questions, so both are
reported here explicitly.

In [6]:
multilabel = sem.multilabel_distribution(data)
display(multilabel.round(2))

print(f"all sentences      : {multilabel.attrs['n_sentences']:,}")
print(f"labelled sentences : {multilabel.attrs['n_labelled_sentences']:,}")
print(f"pct_of_all_sentences sums to      "
      f"{multilabel['pct_of_all_sentences'].sum():.4f}")
print(f"pct_of_labelled_sentences sums to "
      f"{multilabel['pct_of_labelled_sentences'].sum():.4f}")

,labels_per_sentence,n_sentences,pct_of_all_sentences,pct_of_labelled_sentences
0,0,407,5.06,NaN
1,1,4472,55.59,58.56
2,2,2521,31.34,33.01
3,3,568,7.06,7.44
4,4,73,0.91,0.96
5,5,2,0.02,0.03
6,6,1,0.01,0.01


all sentences      : 8,044
labelled sentences : 7,637
pct_of_all_sentences sums to      100.0000
pct_of_labelled_sentences sums to 100.0000


### 3.4 Unlabelled sentences and `Other`

407 sentences (5.1%) carry no label at all. Reading them, they are
scene-setting or procedural - the prompt working as intended. `Other` is 36
assignments (0.3%), which is the taxonomy-coverage result: the frozen scheme
accounts for essentially the whole space.

In [7]:
sentences = data["sentences"]
empty = sentences.loc[sentences["n_labels"].eq(0)]
print(f"unlabelled sentences: {len(empty)} / {len(sentences)} "
      f"({100 * len(empty) / len(sentences):.1f}%)")
print("\nfirst five, for inspection:")
for text in empty["text"].head(5):
    print("  -", text[:110])

other = labels.loc[labels["category"].astype(str).eq(sem.OTHER_CATEGORY)]
print(f"\n'Other' assignments: {len(other)} / {len(labels)} "
      f"({100 * len(other) / len(labels):.2f}%)")
print("descriptions given by the annotator:")
for description in other["other_description"].dropna().head(8):
    print("  -", description)

unlabelled sentences: 407 / 8044 (5.1%)

first five, for inspection:
  - The discussion revolves entirely around card swapping and identifying roles (Robber, Troublemaker, Seer).
  - The discussion centers heavily on role swapping and deception, particularly involving Jacob and Sean.
  - The transcript primarily details role swaps and the identification of several Team Village roles (Drunk, Insom
  - The discussion primarily revolves around identifying the Tanner and Villager roles, with players speculating a
  - The discussion primarily revolves around role confusion, specifically Masons and the Troublemaker, and Drunk m

'Other' assignments: 36 / 11526 (0.31%)
descriptions given by the annotator:
  - Non-player announcer confirmation of role presence
  - role assignment information not attributed to player testimony
  - Factual identification from a transcript rather than a player claim
  - A player's stated suspicion about the possible presence of a Werewolf
  - Strategic urgency an

## 4. Annotation corpus summary

S1. Note the **length differences**: the models do not write justifications of
the same length, which is what makes the sensitivity check in section 5.3
necessary rather than decorative.

In [8]:
summary = sem.annotation_summary(data)
display(summary.round(3))

length = sem.sentence_length_summary(data["justifications"])
display(length.round(3))

,model,decoding_group,n_justifications,n_games,n_sentences,mean_sentences_per_justification,sd_sentences_per_justification,n_labels,mean_labels_per_justification,mean_distinct_categories,pct_no_substantive_category,pct_correct,n_empty_label_sentences,pct_empty_label_sentences,labels_per_sentence
0,Gemma 4 2B,Stochastic,573,191,1808,3.155,0.449,2385,4.162,2.909,0.0,40.663,110,6.084,1.319
1,Gemma 4 2B,Greedy,191,191,599,3.136,0.438,779,4.079,2.874,0.0,36.649,38,6.344,1.301
2,Gemma 4 4B,Stochastic,573,191,2230,3.892,0.483,3114,5.435,3.295,0.0,41.187,111,4.978,1.396
3,Gemma 4 4B,Greedy,191,191,758,3.969,0.422,988,5.173,3.152,0.0,40.838,59,7.784,1.303
4,Gemma 4 31B,Stochastic,573,191,1994,3.480,0.575,3201,5.586,3.382,0.0,41.710,70,3.511,1.605
5,Gemma 4 31B,Greedy,191,191,655,3.429,0.566,1059,5.545,3.414,0.0,43.979,19,2.901,1.617


,model,decoding_group,run_label,n_justifications,n_sentences,mean_sentences,sd_sentences,median_sentences,min_sentences,max_sentences
0,Gemma 4 2B,Stochastic,run_1,191,610,3.194,0.480,3.0,2,5
1,Gemma 4 2B,Stochastic,run_2,191,605,3.168,0.427,3.0,2,5
2,Gemma 4 2B,Stochastic,run_3,191,593,3.105,0.435,3.0,2,4
3,Gemma 4 2B,Greedy,greedy_t0,191,599,3.136,0.438,3.0,2,5
4,Gemma 4 4B,Stochastic,run_1,191,758,3.969,0.502,4.0,3,7
5,Gemma 4 4B,Stochastic,run_2,191,735,3.848,0.462,4.0,3,6
6,Gemma 4 4B,Stochastic,run_3,191,737,3.859,0.477,4.0,3,6
7,Gemma 4 4B,Greedy,greedy_t0,191,758,3.969,0.422,4.0,3,6
8,Gemma 4 31B,Stochastic,run_1,191,662,3.466,0.560,3.0,3,6
9,Gemma 4 31B,Stochastic,run_2,191,669,3.503,0.561,3.0,2,5


## 5. Semantic profile

**Primary metric.** For model *m*, run *r*, game *g*, category *c*:

$$I[m,r,g,c] = 1 \text{ if } c \text{ appears anywhere in that justification}$$
$$P[m,r,c] = \text{mean}_g\, I[m,r,g,c]$$

the share of justifications in which the model invokes the category at least
once.

### 5.1 Run-level prevalence (S2) and the model summary (S3)

In [9]:
run_level = sem.run_level_prevalence(data["justifications"])
prevalence = sem.model_prevalence(run_level)

substantive = prevalence.loc[prevalence["is_substantive"]]
print("Prevalence (% of justifications invoking the category)\n")
display(
    (100 * substantive.pivot_table(
        index="category", columns=["decoding_group", "model"],
        values="prevalence_mean", observed=True)).round(1)
)

print("\nStochastic SD across the three runs (percentage points)\n")
display(
    (100 * substantive.loc[
        substantive["decoding_group"].astype(str).eq("Stochastic")
    ].pivot_table(index="category", columns="model",
                  values="prevalence_sd", observed=True)).round(2)
)

Prevalence (% of justifications invoking the category)



decoding_group  Stochastic                            Greedy                       
model           Gemma 4 2B Gemma 4 4B Gemma 4 31B Gemma 4 2B Gemma 4 4B Gemma 4 31B
category                                                                           
Mechanical             3.0        7.0        52.9        2.6        7.3        53.9
Testimony             53.4       75.4        98.3       54.5       71.7        99.0
SocialJudgment        59.3       56.2        29.8       50.8       53.9        30.4
Behavioral            16.8       58.1        61.6       17.8       50.3        61.3
ClaimComparison        7.9       18.3        57.2        7.9       19.9        59.7
Payoff                90.9       71.9        27.6       90.1       71.7        29.3
Uncertainty           59.7       42.6        10.8       63.9       40.3         7.9


Stochastic SD across the three runs (percentage points)



model,Gemma 4 2B,Gemma 4 4B,Gemma 4 31B
category,,,
Mechanical,0.80,0.80,0.00
Testimony,1.39,1.89,1.32
SocialJudgment,4.86,1.60,3.43
Behavioral,0.91,3.18,0.80
ClaimComparison,1.05,1.89,2.88
Payoff,1.98,1.21,4.07
Uncertainty,0.00,3.20,2.18


### 5.2 `Other`, and the descriptive companions

`Other` is reported once, here, and then dropped from every inferential
analysis.

In [10]:
other_rows = prevalence.loc[~prevalence["is_substantive"]]
display(
    (100 * other_rows.pivot_table(
        index="category", columns=["decoding_group", "model"],
        values="prevalence_mean", observed=True)).round(2)
)

print("mean labels per justification, mean distinct substantive categories,")
print("and share of justifications with no substantive category at all:\n")
display(summary[["model", "decoding_group", "mean_labels_per_justification",
                 "mean_distinct_categories", "pct_no_substantive_category"]]
        .round(3))

decoding_group Stochastic                            Greedy                       
model          Gemma 4 2B Gemma 4 4B Gemma 4 31B Gemma 4 2B Gemma 4 4B Gemma 4 31B
category                                                                          
Other                0.52       0.52        2.79       1.05       1.05        3.14

mean labels per justification, mean distinct substantive categories,
and share of justifications with no substantive category at all:



,model,decoding_group,mean_labels_per_justification,mean_distinct_categories,pct_no_substantive_category
0,Gemma 4 2B,Stochastic,4.162,2.909,0.0
1,Gemma 4 2B,Greedy,4.079,2.874,0.0
2,Gemma 4 4B,Stochastic,5.435,3.295,0.0
3,Gemma 4 4B,Greedy,5.173,3.152,0.0
4,Gemma 4 31B,Stochastic,5.586,3.382,0.0
5,Gemma 4 31B,Greedy,5.545,3.414,0.0


### 5.3 Sensitivity: does normalising for length change anything?

The models differ in length, so prevalence could in principle be inflated for
the more verbose model purely by having more sentences in which a category
might appear. The check: recompute as **assignments per 100 sentences** and
ask whether the model ordering within each category changes.

This stays a sensitivity analysis. It is promoted to a primary result only if
it overturns something. A changed ordering counts as overturning something only
if the primary analysis actually distinguishes those two models for that
category - which section 6.1 checks against the bootstrap intervals.

In [11]:
density = sem.density_sensitivity(data["justifications"], data["labels"])
stochastic_density = density.loc[
    density["decoding_group"].astype(str).eq("Stochastic")]

display(
    stochastic_density.pivot_table(index="category", columns="model",
                                   values="density_mean", observed=True).round(1)
)

# Does the model ordering per category survive the change of metric?
prevalence_rank = substantive.loc[
    substantive["decoding_group"].astype(str).eq("Stochastic")
].pivot_table(index="category", columns="model",
              values="prevalence_mean", observed=True)
density_rank = stochastic_density.pivot_table(
    index="category", columns="model", values="density_mean", observed=True)

rows = []
for category in sem.CATEGORY_ORDER:
    order_prevalence = tuple(prevalence_rank.loc[category].sort_values(
        ascending=False).index)
    order_density = tuple(density_rank.loc[category].sort_values(
        ascending=False).index)
    rows.append({
        "category": category,
        "order_by_prevalence": " > ".join(m.replace("Gemma 4 ", "")
                                          for m in order_prevalence),
        "order_by_density": " > ".join(m.replace("Gemma 4 ", "")
                                       for m in order_density),
        "same_ordering": order_prevalence == order_density,
    })
ordering = pd.DataFrame(rows)
display(ordering)

n_changed = int((~ordering["same_ordering"]).sum())
print(f"\ncategories whose model ordering changes under length "
      f"normalisation: {n_changed} of {len(ordering)}")
for row in ordering.loc[~ordering["same_ordering"]].itertuples():
    print(f"  {row.category}: {row.order_by_prevalence} -> "
          f"{row.order_by_density}")
print("\n-> reconciled against the bootstrap intervals in section 6")

model,Gemma 4 2B,Gemma 4 4B,Gemma 4 31B
category,,,
Mechanical,1.0,2.1,18.6
Testimony,24.5,32.7,67.2
SocialJudgment,31.4,31.3,11.1
Behavioral,6.4,29.5,26.3
ClaimComparison,2.8,7.3,24.5
Payoff,33.0,21.5,8.6
Uncertainty,32.5,15.1,3.3


,category,order_by_prevalence,order_by_density,same_ordering
0,Mechanical,31B > 4B > 2B,31B > 4B > 2B,True
1,Testimony,31B > 4B > 2B,31B > 4B > 2B,True
2,SocialJudgment,2B > 4B > 31B,2B > 4B > 31B,True
3,Behavioral,31B > 4B > 2B,4B > 31B > 2B,False
4,ClaimComparison,31B > 4B > 2B,31B > 4B > 2B,True
5,Payoff,2B > 4B > 31B,2B > 4B > 31B,True
6,Uncertainty,2B > 4B > 31B,2B > 4B > 31B,True



categories whose model ordering changes under length normalisation: 1 of 7
  Behavioral: 31B > 4B > 2B -> 4B > 31B > 2B

-> reconciled against the bootstrap intervals in section 6


## 6. Pairwise model comparisons (S4)

Paired game-level bootstrap. One replicate resamples the 191 game ids with
replacement, uses the **same** ids for every model, recomputes prevalence per
run, averages the three runs into one model value, then differences the models.

`ci_excludes_zero` records a property of the interval. It is **not** a
significance test and no multiple-comparison correction is applied, because no
decision rule is being run.

In [12]:
differences = sem.prevalence_bootstrap_differences(data["justifications"])

stochastic_differences = differences.loc[
    differences["decoding_group"].astype(str).eq("Stochastic")]
display(
    stochastic_differences[
        ["category", "model_a", "model_b", "prevalence_a", "prevalence_b",
         "difference", "ci_low", "ci_high", "ci_excludes_zero"]
    ].round(3).reset_index(drop=True)
)

n_excluding = int(stochastic_differences["ci_excludes_zero"].sum())
print(f"{n_excluding} of {len(stochastic_differences)} stochastic intervals "
      f"exclude zero")

,category,model_a,model_b,prevalence_a,prevalence_b,difference,ci_low,ci_high,ci_excludes_zero
0,Mechanical,Gemma 4 2B,Gemma 4 31B,0.030,0.529,-0.499,-0.553,-0.445,True
1,Mechanical,Gemma 4 2B,Gemma 4 4B,0.030,0.070,-0.040,-0.068,-0.014,True
2,Mechanical,Gemma 4 4B,Gemma 4 31B,0.070,0.529,-0.459,-0.517,-0.401,True
3,Testimony,Gemma 4 2B,Gemma 4 31B,0.534,0.983,-0.449,-0.504,-0.391,True
4,Testimony,Gemma 4 2B,Gemma 4 4B,0.534,0.754,-0.220,-0.272,-0.168,True
5,Testimony,Gemma 4 4B,Gemma 4 31B,0.754,0.983,-0.229,-0.274,-0.183,True
6,SocialJudgment,Gemma 4 2B,Gemma 4 31B,0.593,0.298,0.295,0.229,0.361,True
7,SocialJudgment,Gemma 4 2B,Gemma 4 4B,0.593,0.562,0.031,-0.024,0.086,False
8,SocialJudgment,Gemma 4 4B,Gemma 4 31B,0.562,0.298,0.264,0.199,0.328,True
9,Behavioral,Gemma 4 2B,Gemma 4 31B,0.168,0.616,-0.449,-0.504,-0.393,True


19 of 21 stochastic intervals exclude zero


### 6.1 Reconciling the length-sensitivity check

A model ordering that flips under length normalisation matters only where the
primary analysis claims an ordering in the first place. For each category whose
ordering changed in section 5.3, the pair that swapped is looked up here: if its
bootstrap interval already spans zero, the primary analysis does not separate
those two models for that category, and the flip overturns nothing.

In [13]:
changed = ordering.loc[~ordering["same_ordering"], "category"].tolist()
rows = []
for category in changed:
    by_density = density_rank.loc[category].sort_values(ascending=False).index
    by_prevalence = prevalence_rank.loc[category].sort_values(
        ascending=False).index
    swapped = {m for m, d in zip(by_prevalence, by_density) if m != d}
    match = stochastic_differences.loc[
        stochastic_differences["category"].astype(str).eq(category)
        & stochastic_differences["model_a"].isin(swapped)
        & stochastic_differences["model_b"].isin(swapped)
    ]
    for row in match.itertuples():
        rows.append({
            "category": category,
            "swapped_pair": f"{row.model_a} vs {row.model_b}",
            "difference": row.difference,
            "ci_low": row.ci_low,
            "ci_high": row.ci_high,
            "ci_excludes_zero": row.ci_excludes_zero,
        })

sensitivity_reconciliation = pd.DataFrame(rows)
if len(sensitivity_reconciliation):
    display(sensitivity_reconciliation.round(4))
    unresolved = sensitivity_reconciliation.loc[
        sensitivity_reconciliation["ci_excludes_zero"]]
    if len(unresolved):
        print("WARNING: an ordering flipped for a pair the primary analysis "
              "does separate - length normalisation changes a conclusion")
    else:
        print("every flipped ordering is a pair whose interval already spans "
              "zero -> length normalisation overturns no conclusion")
else:
    print("no ordering changed; nothing to reconcile")

,category,swapped_pair,difference,ci_low,ci_high,ci_excludes_zero
0,Behavioral,Gemma 4 4B vs Gemma 4 31B,-0.0349,-0.0995,0.0314,False


every flipped ordering is a pair whose interval already spans zero -> length normalisation overturns no conclusion


## 7. Semantic co-occurrence

Justification-level binary presence vectors over the seven substantive
categories.

- **joint prevalence** `P(c1, c2)` - the main quantity, the share of
  justifications carrying both;
- **support** - the raw count, kept beside every ratio so a large association
  resting on a handful of justifications is visible;
- **lift** `P(c1,c2) / (P(c1) P(c2))` - secondary and diagnostic only.

Diagonal handling: joint prevalence on the diagonal is the marginal prevalence
and is kept; lift on the diagonal would be `1/P(c)`, an artefact of the
definition, and is set to `NaN`. Lift is `NaN` wherever a marginal is zero.
`Other` is excluded.

In [14]:
co = sem.cooccurrence(data["justifications"])
joint, lift = co["joint"], co["lift"]

for model in sem.MODEL_ORDER:
    print(f"\n{model} - joint prevalence (%), stochastic mean of 3 runs")
    display((100 * sem.cooccurrence_matrix(joint, model, "Stochastic")).round(1))


Gemma 4 2B - joint prevalence (%), stochastic mean of 3 runs


category_b,Mechanical,Testimony,SocialJudgment,Behavioral,ClaimComparison,Payoff,Uncertainty
category_a,,,,,,,
Mechanical,3.0,2.3,1.0,0.5,0.2,2.6,1.2
Testimony,2.3,53.4,29.5,7.7,7.7,48.9,24.8
SocialJudgment,1.0,29.5,59.3,8.7,3.5,55.7,30.2
Behavioral,0.5,7.7,8.7,16.8,1.7,14.7,11.2
ClaimComparison,0.2,7.7,3.5,1.7,7.9,6.5,4.0
Payoff,2.6,48.9,55.7,14.7,6.5,90.9,53.1
Uncertainty,1.2,24.8,30.2,11.2,4.0,53.1,59.7



Gemma 4 4B - joint prevalence (%), stochastic mean of 3 runs


category_b,Mechanical,Testimony,SocialJudgment,Behavioral,ClaimComparison,Payoff,Uncertainty
category_a,,,,,,,
Mechanical,7.0,6.5,2.6,2.8,0.9,5.4,1.7
Testimony,6.5,75.4,39.1,44.5,17.5,55.7,29.3
SocialJudgment,2.6,39.1,56.2,31.6,9.1,41.5,21.6
Behavioral,2.8,44.5,31.6,58.1,12.4,39.6,21.1
ClaimComparison,0.9,17.5,9.1,12.4,18.3,12.9,5.6
Payoff,5.4,55.7,41.5,39.6,12.9,71.9,26.7
Uncertainty,1.7,29.3,21.6,21.1,5.6,26.7,42.6



Gemma 4 31B - joint prevalence (%), stochastic mean of 3 runs


category_b,Mechanical,Testimony,SocialJudgment,Behavioral,ClaimComparison,Payoff,Uncertainty
category_a,,,,,,,
Mechanical,52.9,51.8,12.4,29.8,25.8,12.9,3.3
Testimony,51.8,98.3,29.1,60.7,57.2,26.7,10.6
SocialJudgment,12.4,29.1,29.8,18.2,14.1,5.4,3.7
Behavioral,29.8,60.7,18.2,61.6,39.1,15.7,4.9
ClaimComparison,25.8,57.2,14.1,39.1,57.2,12.7,5.6
Payoff,12.9,26.7,5.4,15.7,12.7,27.6,3.5
Uncertainty,3.3,10.6,3.7,4.9,5.6,3.5,10.8


In [15]:
# Symmetry and diagonal handling, verified rather than assumed.
for model in sem.MODEL_ORDER:
    matrix = sem.cooccurrence_matrix(joint, model, "Stochastic").to_numpy()
    assert np.allclose(matrix, matrix.T), f"{model}: joint matrix not symmetric"

    lift_matrix = sem.cooccurrence_matrix(
        lift, model, "Stochastic", "lift_mean").to_numpy()
    off_diagonal = ~np.eye(len(sem.CATEGORY_ORDER), dtype=bool)
    assert np.allclose(lift_matrix[off_diagonal], lift_matrix.T[off_diagonal],
                       equal_nan=True), f"{model}: lift matrix not symmetric"
    assert np.isnan(np.diag(lift_matrix)).all(), f"{model}: lift diagonal set"

assert not np.isinf(lift["lift_mean"].to_numpy(dtype=float)).any()
print("co-occurrence matrices symmetric; lift diagonal NaN; no infinite lift")

co-occurrence matrices symmetric; lift diagonal NaN; no infinite lift


### 7.1 Ranked pairs, with support in view

Thin-support pairs are **flagged, not dropped** - a pair resting on four
justifications can carry a spectacular lift, and hiding it is worse than
marking it.

In [16]:
pairs = sem.ranked_pairs(joint, lift, "Stochastic", min_support=10.0)

for model in sem.MODEL_ORDER:
    subset = pairs.loc[pairs["model"].astype(str).eq(model)]
    print(f"\n{model} - top 6 pairs by joint prevalence")
    display(subset.head(6)[
        ["pair", "joint_prevalence_mean", "prevalence_a_mean",
         "prevalence_b_mean", "support_mean", "lift_mean", "support_is_thin"]
    ].round(3).reset_index(drop=True))

    print(f"{model} - highest and lowest lift (thin support flagged)")
    extremes = pd.concat([
        subset.nlargest(3, "lift_mean"), subset.nsmallest(3, "lift_mean")
    ])
    display(extremes[["pair", "lift_mean", "joint_prevalence_mean",
                      "support_mean", "support_is_thin"]]
            .round(3).reset_index(drop=True))


Gemma 4 2B - top 6 pairs by joint prevalence


,pair,joint_prevalence_mean,prevalence_a_mean,prevalence_b_mean,support_mean,lift_mean,support_is_thin
0,Payoff + SocialJudgment,0.557,0.909,0.593,106.333,1.033,False
1,Payoff + Uncertainty,0.531,0.909,0.597,101.333,0.978,False
2,Payoff + Testimony,0.489,0.909,0.534,93.333,1.007,False
3,SocialJudgment + Uncertainty,0.302,0.593,0.597,57.667,0.853,False
4,SocialJudgment + Testimony,0.295,0.593,0.534,56.333,0.927,False
5,Testimony + Uncertainty,0.248,0.534,0.597,47.333,0.776,False


Gemma 4 2B - highest and lowest lift (thin support flagged)


,pair,lift_mean,joint_prevalence_mean,support_mean,support_is_thin
0,ClaimComparison + Testimony,1.837,0.077,14.667,False
1,Mechanical + Testimony,1.484,0.023,4.333,True
2,Behavioral + ClaimComparison,1.299,0.017,3.333,True
3,Mechanical + SocialJudgment,0.577,0.010,2.000,True
4,ClaimComparison + Mechanical,0.700,0.002,0.333,True
5,ClaimComparison + SocialJudgment,0.705,0.035,6.667,True



Gemma 4 4B - top 6 pairs by joint prevalence


...

,pair,lift_mean,joint_prevalence_mean,support_mean,support_is_thin
0,ClaimComparison + Testimony,1.267,0.175,33.333,False
1,Mechanical + Testimony,1.235,0.065,12.333,False
2,Behavioral + ClaimComparison,1.149,0.124,23.667,False
3,Mechanical + Uncertainty,0.589,0.017,3.333,True
4,ClaimComparison + Mechanical,0.651,0.009,1.667,True
5,Mechanical + SocialJudgment,0.656,0.026,5.000,True



Gemma 4 31B - top 6 pairs by joint prevalence


,pair,joint_prevalence_mean,prevalence_a_mean,prevalence_b_mean,support_mean,lift_mean,support_is_thin
0,Behavioral + Testimony,0.607,0.616,0.983,116.000,1.003,False
1,ClaimComparison + Testimony,0.572,0.572,0.983,109.333,1.018,False
2,Mechanical + Testimony,0.518,0.529,0.983,99.000,0.998,False
3,Behavioral + ClaimComparison,0.391,0.616,0.572,74.667,1.109,False
4,Behavioral + Mechanical,0.298,0.616,0.529,57.000,0.916,False
5,SocialJudgment + Testimony,0.291,0.298,0.983,55.667,0.992,False


Gemma 4 31B - highest and lowest lift (thin support flagged)


,pair,lift_mean,joint_prevalence_mean,support_mean,support_is_thin
0,Payoff + Uncertainty,1.187,0.035,6.667,True
1,SocialJudgment + Uncertainty,1.111,0.037,7.000,True
2,Behavioral + ClaimComparison,1.109,0.391,74.667,False
3,Mechanical + Uncertainty,0.584,0.033,6.333,True
4,Payoff + SocialJudgment,0.665,0.054,10.333,False
5,Behavioral + Uncertainty,0.723,0.049,9.333,True


## 8. Correctness: overall category-presence association (3A / S7)

For each model and category, over the **three stochastic runs pooled**
(573 outputs per model):

$$\Delta_c = P(\text{correct} \mid c \text{ present}) - P(\text{correct} \mid c \text{ absent})$$

Uncertainty is a **game-level cluster bootstrap**: a game is resampled with all
three of its runs attached.

**Associational.** A positive delta says outputs mentioning the category were
more often correct - not that mentioning it helped. The same game content can
drive both what gets mentioned and whether the vote lands.

In [17]:
association = sem.correctness_presence_association(data["justifications"])
stochastic_association = association.loc[
    association["decoding_group"].astype(str).eq("Stochastic")]

display(
    stochastic_association[
        ["model", "category", "n_present", "n_absent", "n_correct_present",
         "n_correct_absent", "p_correct_present", "p_correct_absent",
         "delta", "ci_low", "ci_high", "ci_excludes_zero"]
    ].round(3).reset_index(drop=True)
)

# Denominators must reconcile: present + absent = 3 runs x 191 games.
assert (stochastic_association["n_present"]
        + stochastic_association["n_absent"] == 573).all()
print("denominators reconcile: n_present + n_absent = 573 for every "
      "model x category")

,model,category,n_present,n_absent,n_correct_present,n_correct_absent,p_correct_present,p_correct_absent,delta,ci_low,ci_high,ci_excludes_zero
0,Gemma 4 2B,Mechanical,17,556,5,228,0.294,0.410,-0.116,-0.321,0.128,False
1,Gemma 4 2B,Testimony,306,267,138,95,0.451,0.356,0.095,-0.012,0.201,False
2,Gemma 4 2B,SocialJudgment,340,233,146,87,0.429,0.373,0.056,-0.047,0.161,False
3,Gemma 4 2B,Behavioral,96,477,34,199,0.354,0.417,-0.063,-0.171,0.053,False
4,Gemma 4 2B,ClaimComparison,45,528,18,215,0.400,0.407,-0.007,-0.162,0.173,False
5,Gemma 4 2B,Payoff,521,52,214,19,0.411,0.365,0.045,-0.111,0.207,False
6,Gemma 4 2B,Uncertainty,342,231,114,119,0.333,0.515,-0.182,-0.279,-0.084,True
7,Gemma 4 4B,Mechanical,40,533,17,219,0.425,0.411,0.014,-0.154,0.216,False
8,Gemma 4 4B,Testimony,432,141,183,53,0.424,0.376,0.048,-0.072,0.164,False
9,Gemma 4 4B,SocialJudgment,322,251,144,92,0.447,0.367,0.081,-0.022,0.183,False


denominators reconcile: n_present + n_absent = 573 for every model x category


## 9. Correctness: stability across games (3B / S8, S9)

For model *m* and game *g*, `K[m,g]` is the number of the three stochastic runs
that voted correctly, so `K` is in {0,1,2,3}. All 191 games are retained.

`Q[m,g,c]` is the share of that game's runs whose justification invokes
category *c*. Comparing mean `Q` across `K` groups asks whether a semantic
basis is more prevalent in games the model solves more consistently - a
property of the games as much as of the model, which is why it stays
descriptive.

In [18]:
groups = sem.correctness_stability_groups(data["justifications"])
display(groups.pivot_table(index="model", columns="label", values="n_games",
                           observed=True))

totals = groups.groupby("model", observed=True)["n_games"].sum()
assert (totals == 191).all(), "K groups do not sum to 191"
print("K groups sum to 191 for every model:", totals.to_dict())

label,0/3 consistently incorrect,1/3,2/3,3/3 consistently correct
model,,,,
Gemma 4 2B,81.0,32.0,33.0,45.0
Gemma 4 4B,84.0,33.0,19.0,55.0
Gemma 4 31B,86.0,27.0,22.0,56.0


K groups sum to 191 for every model: {'Gemma 4 2B': 191, 'Gemma 4 4B': 191, 'Gemma 4 31B': 191}


In [19]:
stability = sem.correctness_stability_semantics(data["justifications"])
print("mean Q (% of a game's runs invoking the category), by K group\n")
display(
    (100 * stability.pivot_table(index=["model", "category"],
                                 columns="k_correct_runs",
                                 values="mean_q", observed=True)).round(1)
)

mean Q (% of a game's runs invoking the category), by K group



k_correct_runs                  0     1      2     3
model       category                                
Gemma 4 2B  Mechanical        3.7   1.0    4.0   2.2
            Testimony        51.0  46.9   52.5  63.0
            SocialJudgment   56.8  60.4   62.6  60.7
            Behavioral       16.0  22.9   16.2  14.1
            ClaimComparison   7.4   7.3   11.1   6.7
            Payoff           93.0  83.3   90.9  92.6
            Uncertainty      65.0  68.8   61.6  42.2
Gemma 4 4B  Mechanical        7.5   4.0   12.3   6.1
            Testimony        73.8  74.7   77.2  77.6
            SocialJudgment   58.3  42.4   43.9  65.5
            Behavioral       63.1  61.6   61.4  47.3
            ClaimComparison  17.1  21.2   21.1  17.6
            Payoff           69.4  67.7   78.9  75.8
            Uncertainty      44.8  48.5   42.1  35.8
Gemma 4 31B Mechanical       50.8  51.9   51.5  57.1
            Testimony        97.7  98.8  100.0  98.2
            SocialJudgment   30.6  24.7   28.8  31.5
            Behavioral       67.1  69.1   63.6  48.8
            ClaimComparison  59.3  51.9   66.7  53.0
            Payoff           26.0  28.4   28.8  29.2
            Uncertainty       9.7  18.5   12.1   8.3

## 10. Correctness: within-game mixed-run contrast (3C / S10)

Restricted to **mixed** model-games, `K = 1` or `K = 2` - the same model
produced both a correct and an incorrect realisation of the same transcript.

$$\Delta^{within}[m,g,c] = \frac{\text{correct runs with } c}{K} - \frac{\text{incorrect runs with } c}{3-K}$$

averaged over mixed games, bootstrapped by resampling mixed model-games with
their complete run sets.

This is the strongest control available without a new experiment: model and
transcript are held fixed and only the sampled realisation varies. It remains
an association between *stated* content and outcome.

In [20]:
contrasts = sem.within_game_contrasts(data["justifications"])
display(
    contrasts[["model", "category", "n_mixed_games",
               "mean_share_correct_runs", "mean_share_incorrect_runs",
               "delta_within", "ci_low", "ci_high", "ci_excludes_zero"]]
    .round(3).reset_index(drop=True)
)

expected_mixed = (groups.loc[groups["k_correct_runs"].isin([1, 2])]
                  .groupby("model", observed=True)["n_games"].sum())
actual_mixed = contrasts.drop_duplicates("model").set_index("model")[
    "n_mixed_games"]
assert (expected_mixed.sort_index() == actual_mixed.sort_index()).all()
print("mixed games = K=1 plus K=2 for every model:", actual_mixed.to_dict())

,model,category,n_mixed_games,mean_share_correct_runs,mean_share_incorrect_runs,delta_within,ci_low,ci_high,ci_excludes_zero
0,Gemma 4 2B,Mechanical,65,0.015,0.038,-0.023,-0.069,0.008,False
1,Gemma 4 2B,Testimony,65,0.531,0.454,0.077,-0.062,0.215,False
2,Gemma 4 2B,SocialJudgment,65,0.638,0.562,0.077,-0.062,0.215,False
3,Gemma 4 2B,Behavioral,65,0.169,0.238,-0.069,-0.185,0.054,False
4,Gemma 4 2B,ClaimComparison,65,0.085,0.100,-0.015,-0.108,0.077,False
5,Gemma 4 2B,Payoff,65,0.892,0.838,0.054,-0.046,0.154,False
6,Gemma 4 2B,Uncertainty,65,0.585,0.715,-0.131,-0.277,0.023,False
7,Gemma 4 4B,Mechanical,52,0.087,0.058,0.029,-0.038,0.106,False
8,Gemma 4 4B,Testimony,52,0.769,0.740,0.029,-0.125,0.192,False
9,Gemma 4 4B,SocialJudgment,52,0.529,0.375,0.154,0.000,0.308,False


mixed games = K=1 plus K=2 for every model: {'Gemma 4 2B': 65, 'Gemma 4 4B': 52, 'Gemma 4 31B': 49}


### 10.1 Do the three correctness analyses agree?

The comparison the plan asks for, laid out per model and category:

- **overall** - present vs absent, pooled over games (section 8);
- **stability** - the 0/3 to 3/3 gradient, `Q(3/3) - Q(0/3)` (section 9);
- **within-game** - correct minus incorrect realisations of the same game
  (section 10).

Agreement in sign across all three is a consistent associational pattern.
A sign that flips between *overall* and *within-game* is the informative case:
it says game-level properties, not the realisation, carry the aggregate
relationship.

In [21]:
gradient = (
    stability.loc[stability["k_correct_runs"].isin([0, 3])]
    .pivot_table(index=["model", "category"], columns="k_correct_runs",
                 values="mean_q", observed=True)
)
gradient["stability_gradient"] = gradient[3] - gradient[0]

comparison = (
    stochastic_association.set_index(["model", "category"])[
        ["delta", "ci_excludes_zero"]]
    .rename(columns={"delta": "overall_delta",
                     "ci_excludes_zero": "overall_ci_excludes_zero"})
    .join(gradient["stability_gradient"])
    .join(contrasts.set_index(["model", "category"])[
        ["delta_within", "ci_excludes_zero"]]
          .rename(columns={"ci_excludes_zero": "within_ci_excludes_zero"}))
)


def sign(value, tolerance=0.02):
    if not np.isfinite(value) or abs(value) < tolerance:
        return "~0"
    return "+" if value > 0 else "-"


comparison["signs"] = [
    f"{sign(o)} {sign(g)} {sign(w)}"
    for o, g, w in zip(comparison["overall_delta"],
                       comparison["stability_gradient"],
                       comparison["delta_within"])
]
comparison["all_three_agree"] = [
    len({sign(o), sign(g), sign(w)}) == 1 and sign(o) != "~0"
    for o, g, w in zip(comparison["overall_delta"],
                       comparison["stability_gradient"],
                       comparison["delta_within"])
]
comparison["overall_within_conflict"] = [
    sign(o) != "~0" and sign(w) != "~0" and sign(o) != sign(w)
    for o, w in zip(comparison["overall_delta"], comparison["delta_within"])
]

display(comparison.round(3))

print("\nconsistent in all three views (|effect| > 0.02):")
for (model, category) in comparison.index[comparison["all_three_agree"]]:
    print(f"  {model:14s} {category}")

print("\noverall and within-game point in opposite directions:")
conflicts = comparison.index[comparison["overall_within_conflict"]]
for (model, category) in conflicts:
    print(f"  {model:14s} {category}")
if not len(conflicts):
    print("  (none)")

overall_delta  overall_ci_excludes_zero  stability_gradient  delta_within  within_ci_excludes_zero     signs  all_three_agree  overall_within_conflict
model       category                                                                                                                                                               
Gemma 4 2B  Mechanical              -0.116                     False              -0.015        -0.023                    False    - ~0 -            False                    False
            Testimony                0.095                     False               0.119         0.077                    False     + + +             True                    False
            SocialJudgment           0.056                     False               0.040         0.077                    False     + + +             True                    False
            Behavioral              -0.063                     False              -0.020        -0.069                    False    - ~0 -            False                    False
            ClaimComparison         -0.007                     False              -0.007        -0.015                    False  ~0 ~0 ~0            False                    False
            Payoff                   0.045                     False              -0.004         0.054                    False    + ~0 +            False                    False
            Uncertainty             -0.182                      True              -0.228        -0.131                    False     - - -             True                    False
Gemma 4 4B  Mechanical               0.014                     False              -0.015         0.029                    False   ~0 ~0 +            False                    False
            Testimony                0.048                     False               0.038         0.029                    False     + + +             True                    False
            SocialJudgment           0.081                     False               0.071         0.154                    False     + + +             True                    False
            Behavioral              -0.130                      True              -0.158        -0.077                    False     - - -             True                    False
            ClaimComparison         -0.003                     False               0.005        -0.029                    False   ~0 ~0 -            False                    False
            Payoff                   0.063                     False               0.063        -0.010                    False    + + ~0            False                    False
            Uncertainty             -0.054                     False              -0.091         0.058                    False     - - +            False                     True
Gemma 4 31B Mechanical               0.025                     False               0.064        -0.082                    False     + + -            False                     True
            Testimony                0.119                     False               0.005         0.010                    False   + ~0 ~0            False                    False
            SocialJudgment           0.039                     False               0.009         0.112                    False    + ~0 +            False                    False
            Behavioral              -0.164                      True              -0.182        -0.112                    False     - - -             True                    False
            ClaimComparison         -0.034                     False              -0.063        -0.010                    False    - - ~0            False                    False
            Payoff                   0.010                     False               0.032        -0.071                    False    ~0 + -            False                    False
            Uncertainty             -0.016                     False 


consistent in all three views (|effect| > 0.02):
  Gemma 4 2B     Testimony
  Gemma 4 2B     SocialJudgment
  Gemma 4 2B     Uncertainty
  Gemma 4 4B     Testimony
  Gemma 4 4B     SocialJudgment
  Gemma 4 4B     Behavioral
  Gemma 4 31B    Behavioral

overall and within-game point in opposite directions:
  Gemma 4 4B     Uncertainty
  Gemma 4 31B    Mechanical


## 11. Greedy robustness checks

Greedy is one deterministic run per game and is reported **separately**
throughout. The question here is only whether it tells the same story.

In [22]:
greedy_prevalence = substantive.loc[
    substantive["decoding_group"].astype(str).eq("Greedy")]
stochastic_prevalence = substantive.loc[
    substantive["decoding_group"].astype(str).eq("Stochastic")]

side_by_side = (
    stochastic_prevalence.set_index(["model", "category"])["prevalence_mean"]
    .rename("stochastic").to_frame()
    .join(greedy_prevalence.set_index(["model", "category"])["prevalence_mean"]
          .rename("greedy"))
)
side_by_side["difference_pp"] = 100 * (side_by_side["greedy"]
                                       - side_by_side["stochastic"])
display(side_by_side.round(3))

print(f"largest |greedy - stochastic| gap: "
      f"{side_by_side['difference_pp'].abs().max():.1f} pp")
print(f"mean |gap|: {side_by_side['difference_pp'].abs().mean():.1f} pp")
print(f"correlation across the 21 model x category cells: "
      f"{side_by_side['stochastic'].corr(side_by_side['greedy']):.4f}")

stochastic  greedy  difference_pp
model       category                                          
Gemma 4 2B  Mechanical            0.030   0.026         -0.349
            Testimony             0.534   0.545          1.047
            SocialJudgment        0.593   0.508         -8.551
            Behavioral            0.168   0.178          1.047
            ClaimComparison       0.079   0.079          0.000
            Payoff                0.909   0.901         -0.873
            Uncertainty           0.597   0.639          4.188
Gemma 4 4B  Mechanical            0.070   0.073          0.349
            Testimony             0.754   0.717         -3.665
            SocialJudgment        0.562   0.539         -2.269
            Behavioral            0.581   0.503         -7.853
            ClaimComparison       0.183   0.199          1.571
            Payoff                0.719   0.717         -0.175
            Uncertainty           0.426   0.403         -2.269
Gemma 4 31B Mechanical            0.529   0.539          1.047
            Testimony             0.983   0.990          0.698
            SocialJudgment        0.298   0.304          0.524
            Behavioral            0.616   0.613         -0.349
            ClaimComparison       0.572   0.597          2.443
            Payoff                0.276   0.293          1.745
            Uncertainty           0.108   0.079         -2.967

largest |greedy - stochastic| gap: 8.6 pp
mean |gap|: 2.1 pp
correlation across the 21 model x category cells: 0.9939


In [23]:
# Greedy pairwise differences and greedy present-vs-absent correctness,
# both kept apart from the stochastic tables above.
greedy_differences = differences.loc[
    differences["decoding_group"].astype(str).eq("Greedy")]
display(greedy_differences[["category", "model_a", "model_b", "difference",
                            "ci_low", "ci_high", "ci_excludes_zero"]]
        .round(3).reset_index(drop=True))

greedy_association = association.loc[
    association["decoding_group"].astype(str).eq("Greedy")]
display(greedy_association[["model", "category", "n_present", "n_absent",
                            "delta", "ci_low", "ci_high", "ci_excludes_zero"]]
        .round(3).reset_index(drop=True))

assert (greedy_association["n_present"]
        + greedy_association["n_absent"] == 191).all()
print("greedy denominators reconcile: n_present + n_absent = 191")

,category,model_a,model_b,difference,ci_low,ci_high,ci_excludes_zero
0,Mechanical,Gemma 4 2B,Gemma 4 31B,-0.513,-0.581,-0.445,True
1,Mechanical,Gemma 4 2B,Gemma 4 4B,-0.047,-0.089,-0.005,True
2,Mechanical,Gemma 4 4B,Gemma 4 31B,-0.466,-0.545,-0.387,True
3,Testimony,Gemma 4 2B,Gemma 4 31B,-0.445,-0.518,-0.372,True
4,Testimony,Gemma 4 2B,Gemma 4 4B,-0.173,-0.251,-0.094,True
5,Testimony,Gemma 4 4B,Gemma 4 31B,-0.272,-0.335,-0.209,True
6,SocialJudgment,Gemma 4 2B,Gemma 4 31B,0.204,0.110,0.298,True
7,SocialJudgment,Gemma 4 2B,Gemma 4 4B,-0.031,-0.115,0.052,False
8,SocialJudgment,Gemma 4 4B,Gemma 4 31B,0.236,0.147,0.325,True
9,Behavioral,Gemma 4 2B,Gemma 4 31B,-0.435,-0.518,-0.346,True


,model,category,n_present,n_absent,delta,ci_low,ci_high,ci_excludes_zero
0,Gemma 4 2B,Mechanical,5,186,0.240,-0.353,0.668,False
1,Gemma 4 2B,Testimony,104,87,0.019,-0.120,0.157,False
2,Gemma 4 2B,SocialJudgment,97,94,0.093,-0.044,0.226,False
3,Gemma 4 2B,Behavioral,34,157,-0.088,-0.259,0.089,False
4,Gemma 4 2B,ClaimComparison,15,176,0.109,-0.161,0.381,False
5,Gemma 4 2B,Payoff,172,19,0.232,0.037,0.401,True
6,Gemma 4 2B,Uncertainty,122,69,-0.243,-0.386,-0.102,True
7,Gemma 4 4B,Mechanical,14,177,-0.055,-0.315,0.224,False
8,Gemma 4 4B,Testimony,137,54,0.053,-0.100,0.205,False
9,Gemma 4 4B,SocialJudgment,103,88,0.188,0.050,0.323,True


greedy denominators reconcile: n_present + n_absent = 191


## 12. Final summary and exported artifacts

In [24]:
tables = sem.build_final_tables(data, REPO_ROOT)
for path in sem.write_final_tables(tables, FINAL_TABLES):
    print("wrote", path.name)

wrote S0_integrity_summary.csv
wrote S0b_repaired_sentences.csv
wrote S0c_multilabel_distribution.csv
wrote S1_annotation_summary.csv
wrote S1_annotation_summary.tex
wrote S1b_sentence_length.csv
wrote S2_run_level_prevalence.csv
wrote S3_model_semantic_prevalence.csv
wrote S3_model_semantic_prevalence.tex
wrote S3b_density_sensitivity.csv
wrote S4_prevalence_bootstrap_differences.csv
wrote S4_prevalence_bootstrap_differences.tex
wrote S5_cooccurrence_joint_prevalence.csv
wrote S5b_cooccurrence_run_level.csv
wrote S6_cooccurrence_lift.csv
wrote S6b_cooccurrence_ranked_pairs.csv
wrote S7_correctness_presence_association.csv
wrote S7_correctness_presence_association.tex
wrote S8_correctness_stability_groups.csv
wrote S9_correctness_stability_semantics.csv
wrote S10_within_game_correctness_contrasts.csv
wrote S10_within_game_correctness_contrasts.tex


In [25]:
for path in figs.build_final_figures(tables, FINAL_FIGURES):
    print("wrote", path.name)

wrote F1_semantic_prevalence.png
wrote F2_semantic_cooccurrence_prevalence.png
wrote F2b_semantic_cooccurrence_lift.png
wrote F3_correctness_presence_association.png
wrote F4_correctness_stability.png
wrote F5_within_game_correctness.png
wrote F2c_semantic_cooccurrence_prevalence_greedy.png
wrote F2d_semantic_cooccurrence_lift_greedy.png
wrote F3b_correctness_presence_greedy.png


### 12.1 Final checks

Everything the analysis plan asks to be verified before the results are
reported, asserted rather than eyeballed.

In [26]:
justifications = data["justifications"]
s7 = tables["S7_correctness_presence_association"]
s8 = tables["S8_correctness_stability_groups"]
s9 = tables["S9_correctness_stability_semantics"]
s10 = tables["S10_within_game_correctness_contrasts"]
s4 = tables["S4_prevalence_bootstrap_differences"]

stochastic_games = sem.presence_tensor(justifications, "Stochastic")[0]
greedy_games = sem.presence_tensor(justifications, "Greedy")[0]

checks = [
    ("corpus is the frozen 2,292 justifications over 191 games",
     len(justifications) == 2292 and justifications["game_id"].nunique() == 191),
    ("no integrity check failed",
     not (tables["S0_integrity_summary"]["status"] == "FAIL").any()),
    ("every model x run has 191 justifications",
     set(tables["S2_run_level_prevalence"]["n_justifications"]) == {191}),
    ("all 191 games present in both decoding groups",
     len(stochastic_games) == 191 and len(greedy_games) == 191),
    ("stochastic averages 3 runs, greedy 1",
     set(tables["S3_model_semantic_prevalence"]
         .groupby("decoding_group", observed=True)["n_runs"]
         .unique().apply(tuple).to_dict().values()) == {(3,), (1,)}),
    ("greedy SD is undefined, never zero",
     tables["S3_model_semantic_prevalence"]
     .loc[lambda f: f["decoding_group"].astype(str).eq("Greedy"),
          "prevalence_sd"].isna().all()),
    ("stochastic and greedy never pooled",
     all(set(t["decoding_group"].astype(str)) <= {"Stochastic", "Greedy"}
         for t in tables.values() if "decoding_group" in t.columns)),
    ("S4 covers 7 categories x 3 pairs x 2 decodings", len(s4) == 42),
    ("every bootstrap CI brackets its point estimate",
     bool(((s4["ci_low"] <= s4["difference"])
           & (s4["difference"] <= s4["ci_high"])).all())
     and bool(((s7["ci_low"] <= s7["delta"])
               & (s7["delta"] <= s7["ci_high"])).all())
     and bool(((s10["ci_low"] <= s10["delta_within"])
               & (s10["delta_within"] <= s10["ci_high"])).all())),
    ("bootstraps are deterministic under the fixed seed",
     sem.prevalence_bootstrap_differences(justifications)["ci_low"]
     .equals(s4["ci_low"])
     and sem.within_game_contrasts(justifications)["ci_low"]
     .equals(s10["ci_low"])),
    ("K groups sum to 191 per model",
     bool((s8.groupby("model", observed=True)["n_games"].sum() == 191).all())),
    ("mixed games equal K=1 plus K=2",
     bool((s10.drop_duplicates("model").set_index("model")["n_mixed_games"]
           .sort_index()
           == s8.loc[s8["k_correct_runs"].isin([1, 2])]
           .groupby("model", observed=True)["n_games"].sum().sort_index()).all())),
    ("stochastic correctness denominators reconcile to 573",
     bool((s7.loc[s7["decoding_group"].astype(str).eq("Stochastic"),
                  "n_present"]
           + s7.loc[s7["decoding_group"].astype(str).eq("Stochastic"),
                    "n_absent"] == 573).all())),
    ("greedy correctness denominators reconcile to 191",
     bool((s7.loc[s7["decoding_group"].astype(str).eq("Greedy"), "n_present"]
           + s7.loc[s7["decoding_group"].astype(str).eq("Greedy"),
                    "n_absent"] == 191).all())),
    ("S9 group sizes sum to 191 per model",
     bool((s9.drop_duplicates(["model", "k_correct_runs"])
           .groupby("model", observed=True)["n_games_in_group"].sum()
           == 191).all())),
    ("co-occurrence excludes Other",
     sem.OTHER_CATEGORY not in set(
         tables["S5_cooccurrence_joint_prevalence"]["category_a"].astype(str))),
    ("lift diagonal is NaN, no infinities anywhere",
     tables["S6_cooccurrence_lift"]
     .loc[lambda f: f["is_diagonal"], "lift_mean"].isna().all()
     and not np.isinf(tables["S6_cooccurrence_lift"]["lift_mean"]
                      .to_numpy(dtype=float)).any()),
    # NOT "no ordering changes" - one does (Behavioral, 4B vs 31B). The claim
    # that matters is that no flip lands on a pair the primary analysis
    # actually separates.
    ("every length-normalisation flip is a pair whose CI spans zero",
     (not bool(sensitivity_reconciliation["ci_excludes_zero"].any()))
     if len(sensitivity_reconciliation) else True),
]

for label, ok in checks:
    print(f"  [{'OK  ' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "a final check failed"
print("\nAll final checks passed.")

  [OK  ] corpus is the frozen 2,292 justifications over 191 games
  [OK  ] no integrity check failed
  [OK  ] every model x run has 191 justifications
  [OK  ] all 191 games present in both decoding groups
  [OK  ] stochastic averages 3 runs, greedy 1
  [OK  ] greedy SD is undefined, never zero
  [OK  ] stochastic and greedy never pooled
  [OK  ] S4 covers 7 categories x 3 pairs x 2 decodings
  [OK  ] every bootstrap CI brackets its point estimate
  [OK  ] bootstraps are deterministic under the fixed seed
  [OK  ] K groups sum to 191 per model
  [OK  ] mixed games equal K=1 plus K=2
  [OK  ] stochastic correctness denominators reconcile to 573
  [OK  ] greedy correctness denominators reconcile to 191
  [OK  ] S9 group sizes sum to 191 per model
  [OK  ] co-occurrence excludes Other
  [OK  ] lift diagonal is NaN, no infinities anywhere
  [OK  ] every length-normalisation flip is a pair whose CI spans zero

All final checks passed.


In [27]:
print(f"tables  -> {FINAL_TABLES.relative_to(REPO_ROOT)}")
for path in sorted(FINAL_TABLES.glob('*.csv')):
    print("   ", path.name)
print(f"\nfigures -> {FINAL_FIGURES.relative_to(REPO_ROOT)}")
for path in sorted(FINAL_FIGURES.glob('*.png')):
    print("   ", path.name)

tables  -> analysis\cross_model\base\voting\prompt_v4\justification_analysis\semantic_annotation\thesis_tables\final_semantic
    S0_integrity_summary.csv
    S0b_repaired_sentences.csv
    S0c_multilabel_distribution.csv
    S10_within_game_correctness_contrasts.csv
    S1_annotation_summary.csv
    S1b_sentence_length.csv
    S2_run_level_prevalence.csv
    S3_model_semantic_prevalence.csv
    S3b_density_sensitivity.csv
    S4_prevalence_bootstrap_differences.csv
    S5_cooccurrence_joint_prevalence.csv
    S5b_cooccurrence_run_level.csv
    S6_cooccurrence_lift.csv
    S6b_cooccurrence_ranked_pairs.csv
    S7_correctness_presence_association.csv
    S8_correctness_stability_groups.csv
    S9_correctness_stability_semantics.csv

figures -> analysis\cross_model\base\voting\prompt_v4\justification_analysis\semantic_annotation\figures\final_semantic
    F1_semantic_prevalence.png
    F2_semantic_cooccurrence_prevalence.png
    F2b_semantic_cooccurrence_lift.png
    F2c_semantic_cooccur